# VietNamNet News — Crawling Data

Notebook crawl dữ liệu bài viết từ VietNamNet cho **19 chuyên mục**.

| Bước | Nội dung |
|------|----------|
| **Section 1** | Chuẩn bị: kiểm tra thư viện, cấu hình, hàm tiện ích |
| **Section 2** | Crawl URL → lưu `Dataset/data_URLs.json` |
| **Section 3** | Crawl title & content → lưu `Dataset/<category>.parquet` |

> **Logic thông minh (per-category):**  
> - Section 2 chỉ crawl URL cho những category **chưa có** trong `data_URLs.json`.  
> - Section 3 chỉ crawl content cho những category **chưa có** file `.parquet`.  
>   Nếu category đó chưa có URL, sẽ tự crawl URL trước rồi mới crawl content.

---
## Section 1: Chuẩn bị

In [ ]:
import importlib

# Kiểm tra thư viện
REQUIRED = {
    "requests":  "requests",
    "tqdm":      "tqdm",
    "bs4":       "beautifulsoup4",
    "lxml":      "lxml",
    "pyarrow":   "pyarrow",
    "newspaper": "newspaper3k",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print("❌ Thiếu các thư viện sau, hãy cài rồi khởi động lại kernel:\n")
    for pkg in missing:
        print(f"    {pkg}")
    print(f"\n⇒ Chạy lệnh:  pip install {' '.join(missing)}")
    raise ImportError(f"Thiếu thư viện: {', '.join(missing)}")

import os
import json
import tqdm
import requests
import pyarrow as pa
import pyarrow.parquet as pq
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import datetime

print("✅ Tất cả thư viện đã sẵn sàng.")


In [ ]:
# Cấu hình
MAX_PAGES   = 500   # Số trang tối đa crawl URL mỗi category
                    # Ví dụ: MAX_PAGES=500 → crawl từ ban-doc đến ban-doc-page499
BATCH_SIZE  = 100   # Số bài ghi vào parquet mỗi lần
NUM_WORKERS = 16     # Số luồng crawl song song

# Đường dẫn
DATASET_DIR    = os.path.normpath(os.path.join(os.getcwd(), "..", "Dataset"))
DATA_URLS_PATH = os.path.join(DATASET_DIR, "data_URLs.json")

# 19 chuyên mục cần crawl
CATEGORIES = [
    "chinh-tri",
    "thoi-su",
    "kinh-doanh",
    "dan-toc-ton-giao",
    "the-thao",
    "giao-duc",
    "the-gioi",
    "doi-song",
    "van-hoa-giai-tri",
    "suc-khoe",
    "cong-nghe",
    "phap-luat",
    "oto-xe-may",
    "du-lich",
    "bat-dong-san",
    "ban-doc",
    "tuan-viet-nam",
    "bao-ve-nguoi-tieu-dung",
    "thi-truong-tieu-dung",
]

print(f"Dataset dir : {DATASET_DIR}")
print(f"URLs file   : {DATA_URLS_PATH}")
print(f"Categories  : {len(CATEGORIES)}")

In [ ]:
# Khởi tạo thư mục Dataset và file data_URLs.json nếu chưa tồn tại
os.makedirs(DATASET_DIR, exist_ok=True)

if not os.path.exists(DATA_URLS_PATH):
    with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
        json.dump({}, f)
    print(f"Đã tạo: {DATA_URLS_PATH}")
else:
    # Nếu file tồn tại nhưng rỗng hoặc lỗi, reset về {}
    try:
        with open(DATA_URLS_PATH, encoding="utf-8") as f:
            content = f.read().strip()
        if not content:
            raise ValueError("File rỗng")
        json.loads(content)  # validate JSON
    except Exception:
        with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
            json.dump({}, f)
        print(f"data_URLs.json bị lỗi/rỗng → đã reset về {{}}: {DATA_URLS_PATH}")

print(f"✅ Dataset dir : {DATASET_DIR}")
print(f"✅ URLs file   : {DATA_URLS_PATH}")

In [ ]:
def get_urls_of_category(category, max_pages=MAX_PAGES):
    """Lấy danh sách URL bài viết của một chuyên mục từ VietNamNet.

    Phạm vi trang:
      page 1  -> https://vietnamnet.vn/{category}
      page 2  -> https://vietnamnet.vn/{category}-page2
      page N-1-> https://vietnamnet.vn/{category}-page{N-1}  (khi max_pages=N)

    Dừng ngay khi gặp trang trống (không có bài nào).
    """
    session = requests.Session()
    urls = []
    for page in tqdm.tqdm(range(1, max_pages), desc=category, leave=False):
        page_url = (
            f"https://vietnamnet.vn/{category}"
            if page == 1
            else f"https://vietnamnet.vn/{category}-page{page}"
        )
        try:
            content = session.get(page_url, timeout=10).content
        except Exception:
            break
        soup = BeautifulSoup(content, "lxml")
        titles = soup.find_all(class_="vnn-title")

        if len(titles) == 0:  # Trang trống -> dung ngay
            break

        for title in titles:
            a = title if title.name == "a" else title.find("a")
            if a:
                href = a.get("href", "")
                if href.startswith("/"):
                    href = "https://vietnamnet.vn" + href
                if href:
                    urls.append(href)
    return urls


def extract_content(url):
    """Trích xuất tiêu đề và nội dung từ một URL bài viết.

    Thứ tự ưu tiên:
      1. BeautifulSoup (bs4) — mặc định, nhanh.
      2. newspaper3k          — fallback khi bs4 thiếu title hoặc content.
    Luôn trả về (title, content) dù có thể rỗng, không bao giờ trả None.
    """
    title, content = "", ""

    # -- Lan 1: BeautifulSoup -------------------------------------------------
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, "lxml")

        title_tag = soup.find("h1")
        if title_tag:
            title = title_tag.get_text(strip=True)

        sapo_tag = soup.find("meta", attrs={"name": "description"})
        sapo = sapo_tag.get("content", "").strip() if sapo_tag else ""

        body_tag = soup.find(class_="main-content")
        paragraphs = [
            p.get_text(strip=True)
            for p in body_tag.find_all("p")
            if p.get_text(strip=True)
        ] if body_tag else []

        content = "\n".join(filter(None, [sapo] + paragraphs))
    except Exception:
        pass

    # -- Lan 2: newspaper3k (fallback khi bs4 thieu title hoac content) -------
    if not title or not content:
        try:
            from newspaper import Article
            article = Article(url, language="vi")
            article.download()
            article.parse()
            if not title:
                title = article.title or ""
            if not content:
                content = article.text or ""
        except Exception:
            pass

    return title, content


SCHEMA = pa.schema([
    pa.field("class",   pa.string()),
    pa.field("title",   pa.string()),
    pa.field("content", pa.string()),
])


def flush_batch(writer, batch):
    """Ghi batch bài viết vào parquet writer rồi xóa batch."""
    if not batch:
        return
    table = pa.table(
        {
            "class":   [r["class"]   for r in batch],
            "title":   [r["title"]   for r in batch],
            "content": [r["content"] for r in batch],
        },
        schema=SCHEMA,
    )
    writer.write_table(table)
    batch.clear()


def fmt_dur(seconds):
    """Format duration: 3661 -> '1h 1m 1s'"""
    s = int(seconds)
    h, rem = divmod(s, 3600)
    m, s   = divmod(rem, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    if m > 0: return f"{m}m {s}s"
    return f"{s}s"


def now():
    """Tra ve gio hien tai dang HH:MM:SS."""
    return datetime.datetime.now().strftime("%H:%M:%S")


print("Da dinh nghia xong cac ham tien ich.")

---
## Section 2: Crawl URL

Crawl danh sách URL bài viết từ VietNamNet và lưu vào `Dataset/data_URLs.json`.

**Chỉ crawl những category chưa có URL trong file.** Category đã có URL sẽ được giữ nguyên.

In [ ]:
# Load dữ liệu URL đã có (nếu có)
if os.path.exists(DATA_URLS_PATH):
    try:
        with open(DATA_URLS_PATH, encoding="utf-8") as f:
            all_urls = json.load(f)
    except Exception:
        all_urls = {}
else:
    all_urls = {}

# Xác định category nào cần crawl URL
missing_url_cats = [
    cat for cat in CATEGORIES
    if cat not in all_urls or len(all_urls[cat]) == 0
]

if not missing_url_cats:
    print("✅ Tất cả categories đã có URL. Bỏ qua bước crawl URL.\n")
    for cat in CATEGORIES:
        print(f"  {cat}: {len(all_urls.get(cat, []))} URLs")
else:
    t0_total = time.time()
    print(f"[{now()}] Cần crawl URL cho {len(missing_url_cats)} category:\n")
    for cat in missing_url_cats:
        print(f"  • {cat}")
    print()

    for i, category in enumerate(missing_url_cats, 1):
        t0 = time.time()
        print(f"[{now()}] [{i}/{len(missing_url_cats)}] Bắt đầu crawl URL: {category}")
        urls = get_urls_of_category(category, max_pages=MAX_PAGES)
        all_urls[category] = urls
        dur = time.time() - t0
        print(f"[{now()}] ✓ {category}: {len(urls)} URLs  ({fmt_dur(dur)})")

    os.makedirs(DATASET_DIR, exist_ok=True)
    with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
        json.dump(all_urls, f, ensure_ascii=False, indent=2)
    total_dur = time.time() - t0_total
    print(f"\n[{now()}] Đã cập nhật URLs vào {DATA_URLS_PATH}")
    print(f"Tổng thời gian crawl URL: {fmt_dur(total_dur)}")

---
## Section 3: Crawl Title và Content

Crawl tiêu đề và nội dung từng bài và lưu thành file `.parquet` trong `Dataset/`.

**Chỉ crawl những category chưa có file `.parquet`.**  
Nếu category đó chưa có URL trong `all_urls`, sẽ tự động crawl URL trước.

In [ ]:
# Xác định category nào cần crawl content
existing_parquet = (
    {f.replace(".parquet", "") for f in os.listdir(DATASET_DIR) if f.endswith(".parquet")}
    if os.path.exists(DATASET_DIR) else set()
)
missing_content_cats = [cat for cat in CATEGORIES if cat not in existing_parquet]

if not missing_content_cats:
    print("✅ Tất cả categories đã có dữ liệu parquet. Bỏ qua bước crawl nội dung.\n")
    for pf in sorted(f for f in os.listdir(DATASET_DIR) if f.endswith(".parquet")):
        print(f"  - {pf}")
else:
    t0_total = time.time()
    print(f"[{now()}] Cần crawl content cho {len(missing_content_cats)} category:\n")
    for cat in missing_content_cats:
        has_url = bool(all_urls.get(cat))
        print(f"  • {cat}  [đã có URL]" if has_url else f"  • {cat}  [chưa có URL, sẽ crawl URL trước]")
    print()

    os.makedirs(DATASET_DIR, exist_ok=True)

    # Đảm bảo all_urls đã được load (nếu chạy section này độc lập)
    if "all_urls" not in dir():
        all_urls = json.load(open(DATA_URLS_PATH, encoding="utf-8")) if os.path.exists(DATA_URLS_PATH) else {}

    for i, category in enumerate(missing_content_cats, 1):
        urls = all_urls.get(category, [])
        prefix = f"[{now()}] [{i}/{len(missing_content_cats)}]"

        # Bước 1: Nếu chưa có URL, crawl URL trước
        if not urls:
            print(f"\n{prefix} {category}: chưa có URL → crawl URL trước...")
            t0 = time.time()
            urls = get_urls_of_category(category, max_pages=MAX_PAGES)
            all_urls[category] = urls
            with open(DATA_URLS_PATH, "w", encoding="utf-8") as f:
                json.dump(all_urls, f, ensure_ascii=False, indent=2)
            print(f"[{now()}] Đã crawl được {len(urls)} URLs  ({fmt_dur(time.time() - t0)})")

        if not urls:
            print(f"[{now()}] Không tìm được URL nào cho {category}, bỏ qua")
            continue

        # Bước 2: Crawl title & content
        print(f"\n{prefix} Bắt đầu crawl content: {category}  ({len(urls)} URLs)")
        t0 = time.time()
        out_path = os.path.join(DATASET_DIR, f"{category}.parquet")
        batch = []
        count = 0

        with pq.ParquetWriter(out_path, SCHEMA) as writer:
            with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
                futures = {executor.submit(extract_content, url): url for url in urls}
                for future in tqdm.tqdm(
                    as_completed(futures), total=len(urls), desc=category
                ):
                    title, content = future.result()
                    if not title and not content:  # bỏ qua nếu cả 2 đều rỗng
                        continue
                    batch.append({"class": category, "title": title, "content": content})
                    count += 1
                    if len(batch) >= BATCH_SIZE:
                        flush_batch(writer, batch)
            flush_batch(writer, batch)

        dur = time.time() - t0
        if count == 0:
            os.remove(out_path)
            print(f"[{now()}] {category}: Không có dữ liệu, bỏ qua")
        else:
            print(f"[{now()}] ✓ {category}: {count} bài viết  ({fmt_dur(dur)})  → {out_path}")

    total_dur = time.time() - t0_total
    elapsed = str(datetime.timedelta(seconds=int(total_dur)))
    print(f"\n✅ Hoàn tất crawl nội dung!  Tổng thời gian: {elapsed}")